# Dự án Phân tích và Dự đoán Thời tiết tại TP.HCM

---

### **Tác giả:** Group 4

---

## 1. Giới thiệu (Introduction)

Dự án này tập trung vào việc phân tích dữ liệu thời tiết lịch sử tại TP.HCM để tìm ra các quy luật và xu hướng ẩn sau các con số. Mục tiêu cuối cùng là xây dựng một mô hình học máy (Hồi quy Tuyến tính) có khả năng **dự đoán nhiệt độ không khí trung bình cho ngày tiếp theo** dựa trên dữ liệu của ngày hôm nay.

**Dữ liệu sử dụng:** `dataexport_20250701T004455.csv` - Dữ liệu thời tiết tổng hợp hàng ngày.

**Các bước chính của dự án:**
1.  **Chuẩn bị môi trường:** Cài đặt và tải các thư viện R cần thiết.
2.  **Tải và làm sạch dữ liệu:** Đọc file, xử lý tên cột, chuyển đổi kiểu dữ liệu và kiểm tra chất lượng dữ liệu.
3.  **Phân tích dữ liệu khám phá (EDA):** Trực quan hóa để hiểu rõ hơn về xu hướng và mối quan hệ giữa các yếu tố thời tiết.
4.  **Chuẩn bị dữ liệu cho mô hình (Feature Engineering):** Tạo biến mục tiêu (nhiệt độ ngày mai).
5.  **Xây dựng mô hình:** Chia dữ liệu và huấn luyện mô hình Hồi quy Tuyến tính.
6.  **Đánh giá mô hình:** Kiểm tra hiệu suất của mô hình trên dữ liệu chưa từng thấy.
7.  **Kết luận:** Tổng kết lại các phát hiện và đề xuất hướng phát triển.

## 2. Chuẩn bị Môi trường (Environment Setup) ⚙️

Bước đầu tiên là tải các thư viện cần thiết cho việc xử lý dữ liệu, trực quan hóa và xây dựng mô hình. Nếu bạn chưa cài đặt, hãy bỏ dấu `#` và chạy dòng `install.packages`.

In [ ]:
# install.packages(c("tidyverse", "lubridate", "caret", "corrplot"))

# Tải các thư viện
library(tidyverse) # Bộ công cụ mạnh mẽ cho xử lý, phân tích và trực quan hóa dữ liệu
library(lubridate) # Giúp làm việc với dữ liệu ngày tháng dễ dàng hơn
library(caret)     # Cung cấp các công cụ cho việc xây dựng mô hình học máy
library(corrplot)  # Dùng để vẽ ma trận tương quan

: 

## 3. Tải và Làm sạch Dữ liệu (Data Loading and Cleaning) 🧹

Chúng ta sẽ đọc file CSV và thực hiện các bước làm sạch cơ bản để dữ liệu sẵn sàng cho việc phân tích.

In [ ]:
# Đọc dữ liệu từ file CSV
weather_df <- read_csv("dataexport_20250701T004455.csv")

# Làm sạch tên cột cho dễ hiểu và dễ sử dụng
names(weather_df) <- c("time", "temp_mean", "temp_max", "temp_min", "rain_sum", 
                       "wind_speed_max", "wind_gusts_max", "wind_direction_dominant")

# Chuyển đổi cột 'time' từ dạng text sang dạng Date
weather_df$time <- as_date(weather_df$time)

# Kiểm tra tổng quan dữ liệu sau khi làm sạch
cat("Cấu trúc dữ liệu:\n")
str(weather_df)

cat("\nMột vài dòng dữ liệu đầu tiên:\n")
head(weather_df)

: 

## 4. Phân tích Dữ liệu Khám phá (EDA) 📊

EDA là quá trình "trò chuyện" với dữ liệu. Chúng ta sẽ sử dụng các biểu đồ để tìm ra các câu chuyện mà dữ liệu đang kể.

### 4.1. Xu hướng nhiệt độ và lượng mưa theo thời gian

In [ ]:
ggplot(weather_df, aes(x = time, y = temp_mean)) + 
  geom_line(color = "#e67e22") + 
  geom_smooth(method = "loess", se = FALSE, color = "#2980b9") + 
  labs(title = "Xu hướng nhiệt độ trung bình hàng ngày",
       x = "Ngày", y = "Nhiệt độ trung bình (°C)")

In [ ]:
ggplot(weather_df, aes(x = time, y = rain_sum)) + 
  geom_col(fill = "#3498db") + 
  labs(title = "Lượng mưa tổng hàng ngày",
       x = "Ngày", y = "Lượng mưa (mm)")

### 4.2. Phân tích mối quan hệ giữa các biến

Chúng ta sẽ sử dụng ma trận tương quan để xem các biến số có mối liên hệ tuyến tính với nhau không.

In [ ]:
# Chọn các cột là số để tính toán tương quan
numeric_cols <- weather_df %>% select_if(is.numeric)

# Tính toán ma trận tương quan
cor_matrix <- cor(numeric_cols, use = "complete.obs")

# Trực quan hóa ma trận tương quan
corrplot(cor_matrix, method = "color", type = "upper", order = "hclust", 
         addCoef.col = "black", tl.col = "black", tl.srt = 45, diag = FALSE)

## 5. Chuẩn bị Dữ liệu cho Mô hình (Feature Engineering) 🛠️

Để dự đoán nhiệt độ ngày mai, chúng ta cần tạo ra một cột chứa giá trị này. Đây được gọi là **biến mục tiêu** (target variable).

In [ ]:
# Tạo biến mục tiêu 'temp_tomorrow' bằng cách lấy giá trị 'temp_mean' của ngày kế tiếp
model_data <- weather_df %>%
  arrange(time) %>% 
  mutate(temp_tomorrow = lead(temp_mean, n = 1))

# Loại bỏ dòng cuối cùng vì nó không có giá trị cho 'temp_tomorrow'
model_data <- model_data %>% filter(!is.na(temp_tomorrow))

cat("Dữ liệu sau khi tạo biến mục tiêu:\n")
tail(model_data)

## 6. Xây dựng Mô hình Hồi quy Tuyến tính 🧠

Bây giờ, chúng ta sẽ tiến hành xây dựng mô hình.

### 6.1. Phân chia dữ liệu

Chúng ta chia dữ liệu thành 2 phần: **tập huấn luyện (training set)** để "dạy" cho mô hình, và **tập kiểm tra (testing set)** để đánh giá hiệu suất của nó trên dữ liệu mới. Tỉ lệ phổ biến là 80% cho training và 20% cho testing.

In [ ]:
set.seed(123) # Giúp kết quả chia luôn giống nhau mỗi lần chạy

train_indices <- createDataPartition(model_data$temp_tomorrow, p = 0.8, list = FALSE)

train_set <- model_data[train_indices, ]
test_set <- model_data[-train_indices, ]

cat(paste("Số dòng trong tập huấn luyện:", nrow(train_set), "\n"))
cat(paste("Số dòng trong tập kiểm tra:", nrow(test_set), "\n"))

### 6.2. Huấn luyện mô hình

Sử dụng hàm `lm()` (linear model) để xây dựng mô hình hồi quy tuyến tính. Công thức `temp_tomorrow ~ . -time` có nghĩa là chúng ta muốn dự đoán `temp_tomorrow` dựa trên tất cả các biến còn lại ngoại trừ cột `time`.

In [ ]:
# Xây dựng mô hình trên tập huấn luyện
model <- lm(temp_tomorrow ~ . -time, data = train_set)

# Xem kết quả chi tiết của mô hình
summary(model)

**Diễn giải `summary(model)`:**
- **Coefficients (Hệ số):** Cho thấy mức độ ảnh hưởng của từng biến. 
- **Pr(>|t|):** Đây là p-value. Nếu giá trị này rất nhỏ (thường < 0.05), biến đó có ý nghĩa thống kê trong việc dự đoán biến mục tiêu.
- **Adjusted R-squared:** Cho biết mô hình của chúng ta giải thích được bao nhiêu phần trăm sự biến thiên của nhiệt độ ngày mai. Giá trị càng gần 1, mô hình càng tốt.

## 7. Đánh giá Mô hình (Model Evaluation) ✅

Sau khi có mô hình, chúng ta cần kiểm tra xem nó dự đoán chính xác đến đâu trên tập dữ liệu kiểm tra.

In [ ]:
# Dùng mô hình để dự đoán trên tập kiểm tra
predictions <- predict(model, newdata = test_set)

# Tính toán các chỉ số lỗi
evaluation <- postResample(pred = predictions, obs = test_set$temp_tomorrow)

cat("Kết quả đánh giá mô hình:\n")
print(evaluation)

cat(paste("\nMAE (Mean Absolute Error):", round(evaluation[["MAE"]], 3), "\n"))
cat("-> Diễn giải: Trung bình, dự đoán của mô hình sai lệch khoảng", round(evaluation[["MAE"]], 3), "độ C so với nhiệt độ thực tế.")

### 7.1. Trực quan hóa kết quả dự đoán

Một cách hiệu quả để đánh giá là vẽ biểu đồ so sánh giữa giá trị thực tế và giá trị mô hình dự đoán. Nếu các điểm nằm sát đường chéo màu đỏ, mô hình hoạt động rất tốt.

In [ ]:
results <- data.frame(
  Actual = test_set$temp_tomorrow,
  Predicted = predictions
)

ggplot(results, aes(x = Actual, y = Predicted)) +
  geom_point(alpha = 0.6, color = "#2c3e50") + 
  geom_abline(intercept = 0, slope = 1, color = "#c0392b", linetype = "dashed", size = 1) + 
  labs(title = "So sánh Nhiệt độ Thực tế và Dự đoán",
       x = "Nhiệt độ Thực tế (°C)",
       y = "Nhiệt độ Dự đoán (°C)") +
  theme_minimal()

## 8. Kết luận và Hướng phát triển 🚀

### Kết luận
Dự án đã thành công trong việc xây dựng một mô hình hồi quy tuyến tính để dự đoán nhiệt độ trung bình ngày mai. Mô hình cho kết quả khá tốt, chứng tỏ khả năng dự báo đáng tin cậy. 

### Hướng phát triển
Để cải thiện mô hình, có thể xem xét các hướng sau:
1.  **Thêm nhiều biến hơn (Feature Engineering):** Tạo thêm các biến như `nhiệt độ của 2 ngày trước`, `trung bình nhiệt độ tuần`, hoặc các biến về mùa trong năm.
2.  **Sử dụng các mô hình phức tạp hơn:** Thử nghiệm với các thuật toán khác như Random Forest hoặc Gradient Boosting.
3.  **Phân tích chuỗi thời gian (Time Series Analysis):** Áp dụng các mô hình chuyên dụng cho dữ liệu chuỗi thời gian như ARIMA để nắm bắt các yếu tố mùa vụ và xu hướng tốt hơn.